# Phase 7: Loss Given Default (LGD) Modeling
This notebook implements Loss Given Default (LGD) modeling. LGD estimates the severity of loss when a loan defaults. Target is defined as: `LGD = (loan_amnt - recoveries) / loan_amnt` capped to `[0.0, 1.0]`.


In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from lgd_model import LGDModel


## 1. Load Data Splits
Load the preprocessed default datasets for model training.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


## 2. Train LGD Models
Instantiate and fit LGD models (XGBoost Regressor and Random Forest benchmark) on historical default occurrences.


In [ ]:
lgd_model = LGDModel()
metrics = lgd_model.fit(train_df, oot_df)
print('Model Evaluation Metrics:')
print(metrics)


## 3. Generate Predictions & Reports
Calculate predictions for the OOT validation set, evaluate error residuals, save the pickled model file, and compile the final PDF model documentation report.


In [ ]:
# Predict on OOT
oot_defaults = oot_df[oot_df['loan_status'].isin(['Charged Off', 'Default'])].copy()
oot_defaults['pred_lgd'] = lgd_model.predict_lgd(oot_defaults)
oot_defaults['actual_lgd'] = lgd_model.calculate_lgd_target(oot_defaults)
print(oot_defaults[['actual_lgd', 'pred_lgd']].describe())

# Save outputs
lgd_model.save_model('outputs/scorecards/lgd_model.pkl')
oot_defaults[['id', 'member_id', 'actual_lgd', 'pred_lgd']].to_csv('outputs/scorecards/lgd_predictions.csv', index=False)
lgd_model.generate_report(train_df, oot_df, 'outputs/reports/lgd_model_report.pdf', metrics)
print('LGD predictions and PDF report generated successfully.')
